File that reads the dataset from the unstructured json file, extracts relevant sections into markdown format, gets the full text of the case, uses regular expressions to split the dataset into "violation of article 10" , "non - violation of article 10" and "other" subsets, picks a subset of the subsets, shuffles it and creates a dataset for manual validation.

## Imports

In [1]:
import pandas as pd
import csv
import sys
from pathlib import Path
import requests
from bs4 import BeautifulSoup
import re
from tqdm import tqdm
tqdm.pandas()

In [2]:
FULL_TEXT_PATH = Path("..") / "dataset" / "text_list.csv"
BASE_URL = 'https://hudoc.echr.coe.int/app/conversion/docx/html/body?library=ECHR&id='
VIOLATIONS_PATH = Path("..") / "dataset" / "violations_echr.csv"
NON_VIOLATIONS_PATH = Path("..") / "dataset" / "non_violations_echr.csv"
ECHR_CLASSIFICATION_PATH = Path("..") / "dataset" / "echr_classification.csv"

## Reading the dataset

Read the unprocessed json of the unstructured cases downloaded from the ECHR OD website

In [3]:
dataset_path = ".." / Path.cwd().parent / "dataset" / "echr_2_0_0_unstructured_cases.json"
df = pd.read_json(dataset_path)

## Functions

Function to extract the relevant sections of the case into markdown format

In [4]:
def extract_section_markdown(case_data:dict, section_name:str) -> str:
    """
    Extracts a specific section from the case data and formats it as markdown.

    Args:
        case_data (dict): The dictionary containing case data.
        section_name (str): The name of the section to extract (e.g., 'law', 'facts').

    Returns:
        str: Formatted markdown text of the specified section.
    """
    
    section_text = ""
    if 'content' in case_data:
        documents = case_data['content']
        for key, sections in documents.items():
            for section in sections:
                if section.get('section_name', '').upper() == section_name.upper():
                    # Append the section title with a heading format
                    section_text += f"### {section['content']}\n\n"
                    section_text += extract_elements(section['elements'])
                    section_text += "\n"

    return section_text.strip()

def extract_elements(elements, indent=0):
    content_text = ""
    for element in elements:
        # Add indentation for each level of nesting
        prefix = "  " * indent
        content_text += f"{prefix}- {element['content']}\n"
        if 'elements' in element:
            # Recursively add further nested elements with increased indentation
            content_text += extract_elements(element['elements'], indent + 1)
    return content_text

Applying the function to populate law, facts, and conclusion columns

In [5]:
df['law'] = df.apply(lambda x: extract_section_markdown(x, 'law'), axis = 1)
df['facts'] = df.apply(lambda x: extract_section_markdown(x, 'facts'), axis = 1)
df['the_conclusion'] = df.apply(lambda x: extract_section_markdown(x, 'conclusion'), axis = 1)

Function to extract html content from a webpage 

In [6]:
def get_full_text_from_html(html_text:str) -> str: 
    """
    Extracts plain text from HTML content by removing scripts, styles, and extra whitespace.

    Args:
        html_text (str): HTML content as a string.

    Returns:
        str: Cleaned plain text with unnecessary elements removed.
    """
    soup = BeautifulSoup(html_text, "html.parser")
    # Remove script and style elements
    for script_or_style in soup(["script", "style"]):
        script_or_style.decompose()

    # Get plain text and replace multiple whitespace characters with a single space
    text = ' '.join(soup.get_text().split())
    return text.replace(u'\xa0', ' ')  # Replace non-breaking spaces

In [7]:
def retrieve_or_fetch_text(dataframe:pd.DataFrame, filename_path:Path, base_url:str)->list:
    """
    Retrieves text data from a CSV file if it exists; otherwise, fetches HTML text 
    from URLs constructed with item IDs in the dataframe.

    Args:
        dataframe (pd.DataFrame): DataFrame containing 'itemid' to construct URLs if CSV is absent.
        filename_path (Path): Path to the CSV file.
        base_url (str): Base URL for constructing the full URL to fetch HTML content.

    Returns:
        list: List of plain text extracted either from the CSV file or from HTML fetched from URLs.
    """
    text_list = []

    # Check if the CSV file exists
    if filename_path.exists():
        # Read text from CSV file
        csv.field_size_limit(sys.maxsize)  # Expand field size for large text
        with open(filename_path, 'r', newline='', encoding='utf-8') as file:
            reader = csv.reader(file)
            for row in reader:
                if row:  # Check if row is not empty
                    full_text = ' '.join(row)
                    text_list.append(full_text)
    else:
        # If the file does not exist, fetch HTML text from URLs
        for itemid in dataframe['itemid']:
            response = requests.get(f"{base_url}{itemid}", timeout=5)
            text_list.append(get_full_text_from_html(response.text))
    
    return text_list


If it not already extracted, extract the textual content from the HTML format of the echr database to get the full text

In [8]:
text_list = retrieve_or_fetch_text(df, FULL_TEXT_PATH, BASE_URL)
df['full_text'] = text_list

Extract the judgement type from the case data

In [9]:
df['judgement_type_1'] = df['documentcollectionid'].apply(lambda x: x[1])
df['judgement_type_2'] = df['documentcollectionid'].apply(lambda x: x[2])

In [10]:
df['judgement_type_1'].value_counts()

judgement_type_1
JUDGMENTS    16096
Name: count, dtype: int64

Slice the dataset to keep only the relevant columns

In [11]:
def slice_dataset(dataset:pd.DataFrame, columns: list[str])->pd.DataFrame:
    """
    Slice the dataset into a subset depending on the names of columns provided

    Args:
        dataset (pd.DataFrame): The original dataset to be sliced
        columns (list[str]): the list of column names that will be kept

    Returns:
        pd.DataFrame: The sliced dataset
    """
    return dataset[columns]

In [12]:
COLUMN_LIST = ['itemid', 'docname','article','appno','judgementdate', 'law', 'facts', 'the_conclusion', 'full_text', 'respondent', 'judgementdate']
echr_df = slice_dataset(df,COLUMN_LIST)

In [13]:
echr_df['the_conclusion'].to_csv('../dataset/the_conclusion.csv')

Split the dataset into violation, non-violation and both at the same time of Article 10

In [14]:
def truncate_text(text, end_phrase='Done in English'):
    """
    Truncates the input text up to and including the end_phrase.
    If the end_phrase is not found, returns the original text.
    """
    
    if not isinstance(text, str):
        return ""  # Or choose to return None or another default value
    
    end_idx = text.lower().find(end_phrase.lower())
    if end_idx != -1:
        end_pos = end_idx + len(end_phrase)
        return text[:end_pos]
    else:
        return text  # Return full text if end_phrase not found

In [15]:
def classify_article10(row):
    text = row['the_conclusion']
    truncated_text = truncate_text(text)

    verb_group = r'\b(Holds|Finds|Declares|Rules|Considers|Decides)\b'
    violation_phrase = r'(violation|violations|breach) of Article 10(?: of the Convention)?'
    
    violation_pattern = rf'(?is){verb_group}.*?\b(has been|have been|is|was|constitutes)\b(?:(?!\bno\b).)*?{violation_phrase}'
    no_violation_pattern = rf'(?is){verb_group}.*?\b(has been|is|was|constitutes)\b.*?\bno\b.*?{violation_phrase}'
    
    # Search for patterns in the text using re.DOTALL to match across newlines
    violation = re.search(violation_pattern, truncated_text, re.DOTALL)
    no_violation = re.search(no_violation_pattern, truncated_text, re.DOTALL)
    
    # Classify based on the presence of patterns
    if violation and no_violation:
        return 'other'
    elif violation:
        return 'violation_of_article_10'
    elif no_violation:
        return 'no_violation_of_article_10'
    else:
        return 'other'


(?is) makes the regex case insensitive and changes the behaviour of the . so that it matches all character including \n <br>
\b ensures we are matching whole words and not substrings within larger words

In [16]:
echr_df['article10_classification'] = echr_df.progress_apply(classify_article10, axis=1)

100%|██████████| 16096/16096 [07:13<00:00, 37.15it/s] 
/var/folders/p0/qc4m31yn4pj7txw3c_cz6gsm0000gn/T/ipykernel_73051/1524134210.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  echr_df['article10_classification'] = echr_df.progress_apply(classify_article10, axis=1)


In [18]:
echr_df.to_csv(ECHR_CLASSIFICATION_PATH)

In [19]:
no_violation_df = echr_df[echr_df['article10_classification'] == 'no_violation_of_article_10']
other_df = echr_df[echr_df['article10_classification'] == 'other']
violation_df = echr_df[echr_df['article10_classification'] == 'violation_of_article_10']

In [20]:
print('Violation cases length is ', len(violation_df))
print('Non-Violation cases length is ', len(no_violation_df))
print('Everything else cases length is ', len(other_df))

Violation cases length is  496
Non-Violation cases length is  142
Everything else cases length is  15458


### Create Random Subset of Dataset

In [21]:
violation_sample = violation_df.sample(n=142, random_state=42) if len(violation_df) >= 142 else violation_df
if len(violation_df) < 142:
    print(f"Warning: Only {len(violation_df)} cases available for 'violation_of_article_10'.")

other_sample = other_df.sample(n=142, random_state=42) if len(other_df) >= 142 else other_df
if len(other_df) < 142:
    print(f"Warning: Only {len(other_df)} cases available for 'other'.")


In [22]:
final_df = pd.concat([no_violation_df,violation_sample, other_sample], ignore_index=True)
final_df

,itemid,docname,article,appno,judgementdate,law,facts,the_conclusion,full_text,respondent,judgementdate,article10_classification
0,001-57513,CASE OF KOSIEK v. GERMANY,[10],9704/82,28/08/1986 00:00:00,### AS TO THE LAW\n\n- I. THE GOVERNMENT’S P...,### AS TO THE FACTS\n\n- 11. Mr. Rolf Kosiek...,"### FOR THESE REASONS, THE COURT\n\n- Holds by...",COURT (PLENARY) CASE OF KOSIEK v. GERMANY (App...,DEU,28/08/1986 00:00:00,no_violation_of_article_10
1,001-192785,CASE OF VESSELINOV v. BULGARIA,[10],3157/16,02/05/2019 00:00:00,### THE LAW\n\n- I. ALLEGED VIOLATION OF ARTI...,### THE FACTS\n\n- I. THE CIRCUMSTANCES OF TH...,"### FOR THESE REASONS, THE COURT, UNANIMOUSLY,...",FIFTH SECTION CASE OF VESSELINOV v. BULGARIA (...,BGR,02/05/2019 00:00:00,no_violation_of_article_10
2,001-57897,CASE OF OTTO-PREMINGER-INSTITUT v. AUSTRIA,"[34, 35, 10]",13470/87,20/09/1994 00:00:00,### AS TO THE LAW\n\n- I. THE GOVERNMENT’S P...,### AS TO THE FACTS\n\n- I. THE PARTICULAR C...,"### FOR THESE REASONS, THE COURT\n\n- 1. Hol...",COURT (CHAMBER) CASE OF OTTO-PREMINGER-INSTITU...,AUT,20/09/1994 00:00:00,no_violation_of_article_10
3,001-179218,CASE OF FRISK AND JENSEN v. DENMARK,[10],19657/12,05/12/2017 00:00:00,### THE LAW\n\n- I. ALLEGED VIOLATION OF ARTI...,### THE FACTS\n\n- I. THE CIRCUMSTANCES OF TH...,"### FOR THESE REASONS, THE COURT, UNANIMOUSLY,...",SECOND SECTION CASE OF FRISK AND JENSEN v. DEN...,DNK,05/12/2017 00:00:00,no_violation_of_article_10
4,001-81066,CASE OF HACHETTE FILIPACCHI ASSOCIES v. FRANCE,[10],71111/01,14/06/2007 00:00:00,### THE LAW\n\n- ALLEGED VIOLATION OF ARTICLE ...,### THE FACTS\n\n- I. THE CIRCUMSTANCES OF TH...,"### FOR THESE REASONS, THE COURT\n\n- Holds by...",FIRST SECTION CASE OF HACHETTE FILIPACCHI ASSO...,FRA,14/06/2007 00:00:00,no_violation_of_article_10
...,...,...,...,...,...,...,...,...,...,...,...,...
421,001-193491,CASE OF ESAMBAYEVA AND OTHERS v. RUSSIA,"[2, 5, 3, 13]",2660/12;2674/12;65488/12;24711/13;24725/13,04/06/2019 00:00:00,### THE LAW\n\n- I. JOINDER OF THE APPLICATIO...,### THE FACTS\n\n- I. THE CIRCUMSTANCES OF TH...,"### FOR THESE REASONS, THE COURT, UNANIMOUSLY,...",THIRD SECTION CASE OF ESAMBAYEVA AND OTHERS v....,RUS,04/06/2019 00:00:00,other
422,001-203168,CASE OF LAVRIK AND OTHERS v. UKRAINE,[5],63542/13;11237/14;20415/14;24962/15;43478/16,25/06/2020 00:00:00,### THE LAW\n\n- JOINDER OF THE APPLICATIONS\n...,### THE FACTS\n\n- 1. The applicants’ persona...,"### FOR THESE REASONS, THE COURT, UNANIMOUSLY,...",FIFTH SECTION CASE OF LAVRIK AND OTHERS v. UKR...,UKR,25/06/2020 00:00:00,other
423,001-186679,CASE OF LEKIĆ v. MONTENEGRO,[6],37726/11,09/10/2018 00:00:00,### THE LAW\n\n- I. LOCUS STANDI OF MS BADEMA...,### THE FACTS\n\n- 5. The applicants were bor...,"### FOR THESE REASONS, THE COURT, UNANIMOUSLY,...",SECOND SECTION CASE OF LEKIĆ v. MONTENEGRO (Ap...,MNE,09/10/2018 00:00:00,other
424,001-101807,CASE OF FABER FIRM AND JAFAROV v. AZERBAIJAN,[6],3365/08,25/11/2010 00:00:00,### THE LAW\n\n- I. ALLEGED VIOLATION OF ARTI...,"### THE FACTS\n\n- 5. The applicant, Mr Sabir...","### FOR THESE REASONS, THE COURT UNANIMOUSLY\n...",FIRST SECTION CASE OF FABER FIRM AND JAFAROV v...,AZE,25/11/2010 00:00:00,other


In [24]:
final_df['Truncated_Text'] = final_df['the_conclusion'].apply(truncate_text)

In [27]:
# Shuffle the dataset 
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

output_df = final_df[['itemid','Truncated_Text', 'article10_classification']].copy()
output_df.rename(columns={'article10_classification': 'Classification'}, inplace=True)

# Add an empty 'Manual_Label' column
output_df['Manual_Label'] = ''

output_df

,itemid,Truncated_Text,Classification,Manual_Label
0,001-97597,"### FOR THESE REASONS, THE COURT UNANIMOUSLY\n...",other,
1,001-194205,"### FOR THESE REASONS, THE COURT, UNANIMOUSLY,...",violation_of_article_10,
2,001-174314,"### FOR THESE REASONS, THE COURT, UNANIMOUSLY,...",no_violation_of_article_10,
3,001-57491,"### FOR THESE REASONS, THE COURT\n\n- 1. Holds...",no_violation_of_article_10,
4,001-215642,"### FOR THESE REASONS, THE COURT, UNANIMOUSLY,...",no_violation_of_article_10,
...,...,...,...,...
421,001-99531,"### FOR THESE REASONS, THE COURT UNANIMOUSLY\n...",other,
422,001-108582,"### FOR THESE REASONS, THE COURT UNANIMOUSLY\n...",no_violation_of_article_10,
423,001-91706,"### FOR THESE REASONS, THE COURT UNANIMOUSLY\n...",no_violation_of_article_10,
424,001-61069,"### FOR THESE REASONS, THE COURT UNANIMOUSLY\n...",other,


In [28]:
output_filename = 'echr_classification_sample.xlsx'
output_df.to_excel(output_filename, index=False)

Out of 21183 English Judgement cases, we have: <br/>
917 violations of article 10 <br/>
261 non violations of article 10 <br/>
19 both violations and non violations of Article 10